# Neural TSP — Distribution-Shift (OOD Generalization) Study

Runs the **variation** experiment with the **trained RL actor** (`actor.pt`):
does the Pointer-Net policy generalize beyond the uniform-random distribution
it was trained on? We evaluate on four point sets — `uniform` (in-distribution
control), `clustered`, `grid`, `ring` — and report a **gap-to-2-Opt** table:
`(neural − 2-Opt) / 2-Opt` per distribution.

A widening gap out-of-distribution (while 2-Opt, which has no training set,
holds steady) is the signal that the policy learned uniform-specific structure
rather than general TSP-solving skill.

**Prerequisites**
- The main training notebook has produced `actor.pt` (in Colab `python/` or
  saved in Drive at `MyDrive/neural-tsp/python/actor.pt`).
- The project copy on Drive includes the new `variation/` folder
  (`generate_ood.py`, `eval_neural_ood.py`, `run_study.py`).

Runtime: **T4 GPU** recommended (neural decoding is the slow part).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1 — Copy project & install deps

Copies the project from Drive to Colab local storage (faster), installs deps,
and checks that the `variation/` folder and a GPU are present.

In [ ]:
import os

# Fresh copy of the local workspace (avoids nested-copy if run more than once)
!rm -rf /content/neural-tsp
!cp -r /content/drive/MyDrive/neural-tsp /content/neural-tsp
%cd /content/neural-tsp

!pip install torch numpy matplotlib pandas -q

# Sanity checks
assert os.path.isdir('/content/neural-tsp/variation'), (
    'variation/ is missing from the Drive copy. Re-upload the project '
    '(including the new variation/ folder) to MyDrive/neural-tsp, or git pull, '
    'then retry.')
print('variation/ contents:', sorted(os.listdir('/content/neural-tsp/variation')))

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (will run on CPU)')

## 2 — Ensure the trained RL `actor.pt` is present

The neural columns need the RL policy `actor.pt` (from `train_rl.py` in the
main notebook). If it isn't already in `python/`, copy it from Drive.

In [ ]:
import os, shutil

actor_local = '/content/neural-tsp/python/actor.pt'
actor_drive = '/content/drive/MyDrive/neural-tsp/python/actor.pt'

if not os.path.exists(actor_local):
    if os.path.exists(actor_drive):
        shutil.copy(actor_drive, actor_local)
        print('Copied actor.pt from Drive -> python/actor.pt')
    else:
        raise FileNotFoundError(
            'actor.pt not found in python/ or Drive. Run the main training '
            'notebook first (train_rl.py saves python/actor.pt).')
else:
    print('actor.pt already present in python/')

print('actor.pt ready:', os.path.exists(actor_local))

## 3 — Run the distribution-shift study

Generates the four datasets, runs the C++ NN/2-Opt baselines on each, runs the
RL policy (greedy + sampling) on each, and assembles the gap-to-2-Opt table as
a DataFrame (`df`) kept in the notebook namespace for the next cell.

Tune `NUM` and `NUM_SAMPLES` to trade speed for precision (e.g. `NUM=200,
NUM_SAMPLES=128` for a quick check).

In [ ]:
import sys
sys.path.insert(0, '/content/neural-tsp/variation')
sys.path.insert(0, '/content/neural-tsp/python')

import torch
from eval_neural_ood import load_policy, evaluate
from run_study import ensure_data, ensure_baselines, run_baselines, DISTRIBUTIONS

# ---- knobs (lower these for a quick check) ----
NUM          = 1000     # instances per distribution
N            = 20
NUM_SAMPLES  = 1280     # tours sampled per instance
TEMPERATURE  = 1.5
# -----------------------------------------------

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

# 1) data (auto-generated + validated if missing)
paths = ensure_data(num=NUM, n=N, seed=0)

# 2) classical baselines (NN + 2-Opt) on every distribution
binp = ensure_baselines()
baseline = {}
print('\n[baselines]')
for name in DISTRIBUTIONS:
    nn_len, two_len = run_baselines(binp, paths[name])
    baseline[name] = {'nn': nn_len, '2opt': two_len}
    print(f'  {name:<11} NN={nn_len:.4f}  2-Opt={two_len:.4f}')

# 3) neural policy (RL actor.pt)
model, src = load_policy(device=device)
assert src == 'actor.pt', f'Expected actor.pt, got {src!r} (check cell 2).'
model.to(device).eval()
print(f'\nLoaded policy: {src}')

neural = {}
for name in DISTRIBUTIONS:
    print(f'  decoding {name} ...')
    res = evaluate(paths[name], model, device=device,
                   num_samples=NUM_SAMPLES, temperature=TEMPERATURE, verbose=True)
    neural[name] = res

# 4) assemble gap-to-2-Opt table
import pandas as pd
rows = []
for name in DISTRIBUTIONS:
    two = baseline[name]['2opt']
    g, s = neural[name]['greedy'], neural[name]['sampling']
    rows.append({
        'distribution':    name,
        'NN':              round(baseline[name]['nn'], 4),
        '2-Opt':           round(two, 4),
        'Neural-Greedy':   round(g, 4),
        'Neural-Sampling': round(s, 4),
        'gap_greedy_%':    round((g - two) / two * 100, 2),
        'gap_sampling_%':  round((s - two) / two * 100, 2),
    })
df = pd.DataFrame(rows)
df

## 4 — Visualize: gap to 2-Opt across distributions

The headline chart. Bars are `(neural − 2-Opt) / 2-Opt` per distribution.
Near 0% on `uniform` is expected (the policy trained there); growth toward the
right indicates distribution-specific overfitting. Saved to Drive for the report.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
x = list(range(len(df)))
w = 0.4
ax.bar([i - w / 2 for i in x], df['gap_greedy_%'],   width=w, label='Greedy')
ax.bar([i + w / 2 for i in x], df['gap_sampling_%'], width=w, label='Sampling')
ax.axhline(0, color='k', linewidth=0.8)
ax.set_xticks(x); ax.set_xticklabels(df['distribution'])
ax.set_ylabel('(neural − 2-Opt) / 2-Opt  [%]')
ax.set_title('Distribution-shift: neural gap to 2-Opt (lower = better)')
ax.legend()
fig.tight_layout()

out = '/content/drive/MyDrive/neural-tsp/variation_ood_gap.png'
fig.savefig(out, dpi=150)
print('Saved figure ->', out)
plt.show()

## 5 — Save results to Drive

Persists the table (CSV) and generated OOD datasets so the run is reproducible
and report-ready without re-running.

In [ ]:
import os, shutil

os.makedirs('/content/drive/MyDrive/neural-tsp/variation_results', exist_ok=True)

csv_path = '/content/drive/MyDrive/neural-tsp/variation_results/ood_gap_table.csv'
df.to_csv(csv_path, index=False)
print('Saved table ->', csv_path)

# back up the generated OOD datasets
shutil.copytree('/content/neural-tsp/variation/data',
                '/content/drive/MyDrive/neural-tsp/variation_results/data',
                dirs_exist_ok=True)
print('Backed up OOD datasets -> variation_results/data/')

## How to read the result

- **`uniform`** is the in-distribution control — the sampling gap should sit
  near 0%, since the policy was trained on this distribution.
- **`clustered` / `grid` / `ring`** are out-of-distribution. If the sampling
  gap grows here while 2-Opt stays flat, the policy has learned uniform-specific
  structure rather than general TSP skill — the study's contribution.
- 2-Opt is distribution-agnostic local search; its gap is 0 by construction.
  NN's gap shows how improvable each distribution is by local search (note
  `ring` is ~0% — on a circle, even NN is already optimal).

Notes
- Absolute tour length is **not** comparable across rows (clustered points are
  closer together → shorter tours for every method). Compare only the
  `gap_*_%` columns across distributions.
- For a fast sanity check, re-run cell 3 with `NUM=200`, `NUM_SAMPLES=128`.